### Imports

In [1]:
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

from datasets import load_dataset
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

### Reproducibility

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


### Common helper functions

In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return clean_text(text).split()


def build_vocab(token_lists, max_vocab):
    counter = Counter()
    for tokens in token_lists:
        counter.update(tokens)

    vocab = {
        word: i + 2
        for i, (word, _) in enumerate(counter.most_common(max_vocab))
    }
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab


def encode_tokens(token_lists, vocab):
    return [[vocab.get(tok, vocab["<UNK>"]) for tok in tokens] for tokens in token_lists]


def pad_sequences_custom(sequences, max_len):
    padded = []
    for seq in sequences:
        seq = seq[:max_len]
        seq = seq + [0] * (max_len - len(seq))
        padded.append(seq)
    return padded


class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout1 = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout1(x)
        _, (h, _) = self.lstm(x)
        h = h[-1]
        h = self.dropout2(h)
        return self.fc(h)


class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout1 = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout2 = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.dropout1(x)
        _, (h, _) = self.lstm(x)
        h_forward = h[-2]
        h_backward = h[-1]
        h_cat = torch.cat((h_forward, h_backward), dim=1)
        h_cat = self.dropout2(h_cat)
        return self.fc(h_cat)


def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate_model(model, loader, device):
    model.eval()
    preds = []
    targets = []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            out = model(X)
            pred = torch.argmax(out, dim=1).cpu().numpy()

            preds.extend(pred)
            targets.extend(y.numpy())

    return {
        "f1_micro": f1_score(targets, preds, average="micro"),
        "f1_macro": f1_score(targets, preds, average="macro"),
        "f1_weighted": f1_score(targets, preds, average="weighted"),
    }


def train_and_select_best(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs,
    save_path
):
    history = []
    best_val_f1 = 0.0

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate_model(model, val_loader, device)

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_f1_micro": val_metrics["f1_micro"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_f1_weighted": val_metrics["f1_weighted"],
        }
        history.append(row)

        print(
            f"Epoch {epoch + 1}: "
            f"loss={train_loss:.4f}, "
            f"val_f1_micro={val_metrics['f1_micro']:.4f}, "
            f"val_f1_macro={val_metrics['f1_macro']:.4f}, "
            f"val_f1_weighted={val_metrics['f1_weighted']:.4f}"
        )

        if val_metrics["f1_micro"] > best_val_f1:
            best_val_f1 = val_metrics["f1_micro"]
            torch.save(model.state_dict(), save_path)

    return pd.DataFrame(history)

### Dataset preparation helper

In [4]:
def prepare_text_data(
    train_texts,
    train_labels,
    test_texts,
    test_labels,
    max_vocab,
    max_len,
    batch_size,
    val_size=0.1,
    random_state=42,
):
    train_texts_split, val_texts, train_labels_split, val_labels = train_test_split(
        train_texts,
        train_labels,
        test_size=val_size,
        random_state=random_state,
        stratify=train_labels
    )

    train_tokens = [tokenize(t) for t in train_texts_split]
    val_tokens = [tokenize(t) for t in val_texts]
    test_tokens = [tokenize(t) for t in test_texts]

    vocab = build_vocab(train_tokens, max_vocab=max_vocab)

    train_encoded = encode_tokens(train_tokens, vocab)
    val_encoded = encode_tokens(val_tokens, vocab)
    test_encoded = encode_tokens(test_tokens, vocab)

    train_padded = pad_sequences_custom(train_encoded, max_len=max_len)
    val_padded = pad_sequences_custom(val_encoded, max_len=max_len)
    test_padded = pad_sequences_custom(test_encoded, max_len=max_len)

    train_loader = DataLoader(
        TextDataset(train_padded, train_labels_split),
        batch_size=batch_size,
        shuffle=True
    )
    val_loader = DataLoader(
        TextDataset(val_padded, val_labels),
        batch_size=batch_size,
        shuffle=False
    )
    test_loader = DataLoader(
        TextDataset(test_padded, test_labels),
        batch_size=batch_size,
        shuffle=False
    )

    return {
        "vocab": vocab,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "train_size": len(train_texts_split),
        "val_size": len(val_texts),
        "test_size": len(test_texts),
    }

## Emotion

### Data loading

In [5]:
emotion_dataset = load_dataset("emotion")

emotion_train_texts = list(emotion_dataset["train"]["text"])
emotion_train_labels = list(emotion_dataset["train"]["label"])

emotion_test_texts = list(emotion_dataset["test"]["text"])
emotion_test_labels = list(emotion_dataset["test"]["label"])

print("Train:", len(emotion_train_texts))
print("Test:", len(emotion_test_texts))

'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера' thrown while requesting HEAD https://huggingface.co/datasets/emotion/resolve/cab853a1dbdf4c42c2b3ef2173804746df8825fe/emotion.py
Retrying in 1s [Retry 1/5].
'[WinError 10060] Попытка установить соединение была безуспешной, т.к. от другого компьютера за требуемое время не получен нужный отклик, или было разорвано уже установленное соединение из-за неверного отклика уже подключенного компьютера' thrown while requesting HEAD https://huggingface.co/datasets/emotion/resolve/cab853a1dbdf4c42c2b3ef2173804746df8825fe/emotion.py
Retrying in 2s [Retry 2/5].


Train: 16000
Test: 2000


### Data preparation

In [6]:
EMOTION_MAX_VOCAB = 10000
EMOTION_MAX_LEN = 50
BATCH_SIZE = 64

emotion_data = prepare_text_data(
    train_texts=emotion_train_texts,
    train_labels=emotion_train_labels,
    test_texts=emotion_test_texts,
    test_labels=emotion_test_labels,
    max_vocab=EMOTION_MAX_VOCAB,
    max_len=EMOTION_MAX_LEN,
    batch_size=BATCH_SIZE,
    val_size=0.1,
    random_state=SEED,
)

print("Emotion vocab size:", len(emotion_data["vocab"]))
print("Train split:", emotion_data["train_size"])
print("Val split:", emotion_data["val_size"])
print("Test size:", emotion_data["test_size"])

Emotion vocab size: 10002
Train split: 14400
Val split: 1600
Test size: 2000


### Emotion — LSTM

In [7]:
emotion_lstm = LSTMModel(
    vocab_size=len(emotion_data["vocab"]),
    embed_dim=128,
    hidden_dim=128,
    num_classes=6,
    dropout=0.3
).to(device)

emotion_lstm_criterion = nn.CrossEntropyLoss()
emotion_lstm_optimizer = torch.optim.Adam(emotion_lstm.parameters(), lr=3e-4)

emotion_lstm_history = train_and_select_best(
    model=emotion_lstm,
    train_loader=emotion_data["train_loader"],
    val_loader=emotion_data["val_loader"],
    optimizer=emotion_lstm_optimizer,
    criterion=emotion_lstm_criterion,
    device=device,
    epochs=10,
    save_path="emotion_best_lstm.pt"
)

emotion_lstm_history

Epoch 1: loss=1.6230, val_f1_micro=0.3362, val_f1_macro=0.0864, val_f1_weighted=0.1730
Epoch 2: loss=1.5792, val_f1_micro=0.3350, val_f1_macro=0.0843, val_f1_weighted=0.1692
Epoch 3: loss=1.5773, val_f1_micro=0.3344, val_f1_macro=0.0842, val_f1_weighted=0.1690
Epoch 4: loss=1.5769, val_f1_micro=0.3344, val_f1_macro=0.0842, val_f1_weighted=0.1690
Epoch 5: loss=1.5738, val_f1_micro=0.3344, val_f1_macro=0.0848, val_f1_weighted=0.1701
Epoch 6: loss=1.5738, val_f1_micro=0.3337, val_f1_macro=0.0841, val_f1_weighted=0.1689
Epoch 7: loss=1.5720, val_f1_micro=0.3337, val_f1_macro=0.0841, val_f1_weighted=0.1689
Epoch 8: loss=1.5691, val_f1_micro=0.3319, val_f1_macro=0.0845, val_f1_weighted=0.1694
Epoch 9: loss=1.5422, val_f1_micro=0.3337, val_f1_macro=0.0872, val_f1_weighted=0.1715
Epoch 10: loss=1.4747, val_f1_micro=0.3738, val_f1_macro=0.1848, val_f1_weighted=0.2598


,epoch,train_loss,val_f1_micro,val_f1_macro,val_f1_weighted
0,1,1.623028,0.336250,0.086420,0.172977
1,2,1.579184,0.335000,0.084278,0.169216
2,3,1.577327,0.334375,0.084160,0.168978
3,4,1.576948,0.334375,0.084161,0.168980
4,5,1.573770,0.334375,0.084827,0.170136
5,6,1.573847,0.333750,0.084121,0.168899
6,7,1.572017,0.333750,0.084121,0.168899
7,8,1.569097,0.331875,0.084459,0.169400
8,9,1.542174,0.333750,0.087163,0.171458
9,10,1.474660,0.373750,0.184841,0.259812


### Emotion — BiLSTM

In [8]:
emotion_bilstm = BiLSTMModel(
    vocab_size=len(emotion_data["vocab"]),
    embed_dim=128,
    hidden_dim=128,
    num_classes=6,
    dropout=0.3
).to(device)

emotion_bilstm_criterion = nn.CrossEntropyLoss()
emotion_bilstm_optimizer = torch.optim.Adam(emotion_bilstm.parameters(), lr=3e-4)

emotion_bilstm_history = train_and_select_best(
    model=emotion_bilstm,
    train_loader=emotion_data["train_loader"],
    val_loader=emotion_data["val_loader"],
    optimizer=emotion_bilstm_optimizer,
    criterion=emotion_bilstm_criterion,
    device=device,
    epochs=10,
    save_path="emotion_best_bilstm.pt"
)

emotion_bilstm_history

Epoch 1: loss=1.5974, val_f1_micro=0.3438, val_f1_macro=0.1227, val_f1_weighted=0.2361
Epoch 2: loss=1.5676, val_f1_micro=0.4012, val_f1_macro=0.1637, val_f1_weighted=0.3091
Epoch 3: loss=1.5157, val_f1_micro=0.4512, val_f1_macro=0.1801, val_f1_weighted=0.3412
Epoch 4: loss=1.4065, val_f1_micro=0.5406, val_f1_macro=0.3108, val_f1_weighted=0.4771
Epoch 5: loss=1.2633, val_f1_micro=0.6156, val_f1_macro=0.3913, val_f1_weighted=0.5640
Epoch 6: loss=1.1014, val_f1_micro=0.7069, val_f1_macro=0.4801, val_f1_weighted=0.6605
Epoch 7: loss=0.9746, val_f1_micro=0.7312, val_f1_macro=0.4979, val_f1_weighted=0.6844
Epoch 8: loss=0.8581, val_f1_micro=0.7588, val_f1_macro=0.5404, val_f1_weighted=0.7163
Epoch 9: loss=0.7676, val_f1_micro=0.7688, val_f1_macro=0.5471, val_f1_weighted=0.7256
Epoch 10: loss=0.6980, val_f1_micro=0.7944, val_f1_macro=0.6030, val_f1_weighted=0.7568


,epoch,train_loss,val_f1_micro,val_f1_macro,val_f1_weighted
0,1,1.597375,0.343750,0.122659,0.236058
1,2,1.567589,0.401250,0.163659,0.309079
2,3,1.515682,0.451250,0.180129,0.341246
3,4,1.406530,0.540625,0.310820,0.477115
4,5,1.263308,0.615625,0.391285,0.564047
5,6,1.101358,0.706875,0.480085,0.660464
6,7,0.974611,0.731250,0.497861,0.684425
7,8,0.858098,0.758750,0.540424,0.716322
8,9,0.767562,0.768750,0.547117,0.725635
9,10,0.697960,0.794375,0.603003,0.756808


### Emotion — Final test results

In [9]:
emotion_lstm.load_state_dict(torch.load("emotion_best_lstm.pt", map_location=device))
emotion_bilstm.load_state_dict(torch.load("emotion_best_bilstm.pt", map_location=device))

emotion_lstm_test = evaluate_model(emotion_lstm, emotion_data["test_loader"], device)
emotion_bilstm_test = evaluate_model(emotion_bilstm, emotion_data["test_loader"], device)

emotion_results = pd.DataFrame([
    {"dataset": "emotion", "model": "LSTM", **emotion_lstm_test},
    {"dataset": "emotion", "model": "BiLSTM", **emotion_bilstm_test},
]).sort_values("f1_micro", ascending=False)

emotion_results

,dataset,model,f1_micro,f1_macro,f1_weighted
1,emotion,BiLSTM,0.8065,0.614334,0.774073
0,emotion,LSTM,0.3910,0.190763,0.276482


## 20_newsgroups (4 classes)

### Data loading and filtering

In [10]:
news_dataset = load_dataset("SetFit/20_newsgroups")

categories = [
    "comp.sys.ibm.pc.hardware",
    "comp.sys.mac.hardware",
    "comp.graphics",
    "comp.windows.x"
]

news_train = news_dataset["train"].filter(lambda x: x["label_text"] in categories)
news_test = news_dataset["test"].filter(lambda x: x["label_text"] in categories)

news_train_texts = list(news_train["text"])
news_test_texts = list(news_test["text"])

news_train_label_texts = list(news_train["label_text"])
news_test_label_texts = list(news_test["label_text"])

label2id = {label: i for i, label in enumerate(categories)}
id2label = {i: label for label, i in label2id.items()}

news_train_labels = [label2id[label] for label in news_train_label_texts]
news_test_labels = [label2id[label] for label in news_test_label_texts]

print("Train:", len(news_train_texts))
print("Test:", len(news_test_texts))
print(label2id)

Repo card metadata block was not found. Setting CardData to empty.


Filter:   0%|          | 0/11314 [00:00<?, ? examples/s]

Filter:   0%|          | 0/7532 [00:00<?, ? examples/s]

Train: 2345
Test: 1561
{'comp.sys.ibm.pc.hardware': 0, 'comp.sys.mac.hardware': 1, 'comp.graphics': 2, 'comp.windows.x': 3}


### Data preparation

In [11]:
NEWS_MAX_VOCAB = 20000
NEWS_MAX_LEN = 300
BATCH_SIZE = 64

news_data = prepare_text_data(
    train_texts=news_train_texts,
    train_labels=news_train_labels,
    test_texts=news_test_texts,
    test_labels=news_test_labels,
    max_vocab=NEWS_MAX_VOCAB,
    max_len=NEWS_MAX_LEN,
    batch_size=BATCH_SIZE,
    val_size=0.1,
    random_state=SEED,
)

print("News vocab size:", len(news_data["vocab"]))
print("Train split:", news_data["train_size"])
print("Val split:", news_data["val_size"])
print("Test size:", news_data["test_size"])

News vocab size: 18489
Train split: 2110
Val split: 235
Test size: 1561


### 20_newsgroups — LSTM

In [12]:
news_lstm = LSTMModel(
    vocab_size=len(news_data["vocab"]),
    embed_dim=200,
    hidden_dim=128,
    num_classes=4,
    dropout=0.3
).to(device)

news_lstm_criterion = nn.CrossEntropyLoss()
news_lstm_optimizer = torch.optim.Adam(news_lstm.parameters(), lr=3e-4)

news_lstm_history = train_and_select_best(
    model=news_lstm,
    train_loader=news_data["train_loader"],
    val_loader=news_data["val_loader"],
    optimizer=news_lstm_optimizer,
    criterion=news_lstm_criterion,
    device=device,
    epochs=8,
    save_path="news_best_lstm.pt"
)

news_lstm_history

Epoch 1: loss=1.3875, val_f1_micro=0.2468, val_f1_macro=0.1081, val_f1_weighted=0.1084
Epoch 2: loss=1.3839, val_f1_micro=0.2511, val_f1_macro=0.1162, val_f1_weighted=0.1164
Epoch 3: loss=1.3829, val_f1_micro=0.2468, val_f1_macro=0.1077, val_f1_weighted=0.1082
Epoch 4: loss=1.3795, val_f1_micro=0.2340, val_f1_macro=0.0965, val_f1_weighted=0.0969
Epoch 5: loss=1.3773, val_f1_micro=0.2468, val_f1_macro=0.1077, val_f1_weighted=0.1082
Epoch 6: loss=1.3743, val_f1_micro=0.2383, val_f1_macro=0.1046, val_f1_weighted=0.1051
Epoch 7: loss=1.3722, val_f1_micro=0.2426, val_f1_macro=0.0993, val_f1_weighted=0.0997
Epoch 8: loss=1.3674, val_f1_micro=0.2426, val_f1_macro=0.1135, val_f1_weighted=0.1138


,epoch,train_loss,val_f1_micro,val_f1_macro,val_f1_weighted
0,1,1.387520,0.246809,0.108065,0.108387
1,2,1.383890,0.251064,0.116225,0.116450
2,3,1.382853,0.246809,0.107715,0.108173
3,4,1.379548,0.234043,0.096491,0.096902
4,5,1.377271,0.246809,0.107715,0.108173
5,6,1.374350,0.238298,0.104643,0.105089
6,7,1.372231,0.242553,0.099303,0.099726
7,8,1.367394,0.242553,0.113460,0.113799


### 20_newsgroups — BiLSTM

In [13]:
news_bilstm = BiLSTMModel(
    vocab_size=len(news_data["vocab"]),
    embed_dim=200,
    hidden_dim=128,
    num_classes=4,
    dropout=0.3
).to(device)

news_bilstm_criterion = nn.CrossEntropyLoss()
news_bilstm_optimizer = torch.optim.Adam(news_bilstm.parameters(), lr=3e-4)

news_bilstm_history = train_and_select_best(
    model=news_bilstm,
    train_loader=news_data["train_loader"],
    val_loader=news_data["val_loader"],
    optimizer=news_bilstm_optimizer,
    criterion=news_bilstm_criterion,
    device=device,
    epochs=8,
    save_path="news_best_bilstm.pt"
)

news_bilstm_history

Epoch 1: loss=1.3863, val_f1_micro=0.2553, val_f1_macro=0.2442, val_f1_weighted=0.2442
Epoch 2: loss=1.3740, val_f1_micro=0.2723, val_f1_macro=0.2640, val_f1_weighted=0.2640
Epoch 3: loss=1.3607, val_f1_micro=0.2936, val_f1_macro=0.2937, val_f1_weighted=0.2936
Epoch 4: loss=1.3509, val_f1_micro=0.3064, val_f1_macro=0.3054, val_f1_weighted=0.3053
Epoch 5: loss=1.3400, val_f1_micro=0.3362, val_f1_macro=0.3341, val_f1_weighted=0.3341
Epoch 6: loss=1.3265, val_f1_micro=0.3404, val_f1_macro=0.3382, val_f1_weighted=0.3381
Epoch 7: loss=1.3079, val_f1_micro=0.3447, val_f1_macro=0.3407, val_f1_weighted=0.3406
Epoch 8: loss=1.2846, val_f1_micro=0.3489, val_f1_macro=0.3489, val_f1_weighted=0.3489


,epoch,train_loss,val_f1_micro,val_f1_macro,val_f1_weighted
0,1,1.386349,0.255319,0.244188,0.244231
1,2,1.374021,0.272340,0.264038,0.264048
2,3,1.360742,0.293617,0.293689,0.293647
3,4,1.350890,0.306383,0.305355,0.305329
4,5,1.340024,0.336170,0.334112,0.334116
5,6,1.326510,0.340426,0.338161,0.338147
6,7,1.307910,0.344681,0.340692,0.340576
7,8,1.284555,0.348936,0.348866,0.348876


### 20_newsgroups — Final test results

In [14]:
news_lstm.load_state_dict(torch.load("news_best_lstm.pt", map_location=device))
news_bilstm.load_state_dict(torch.load("news_best_bilstm.pt", map_location=device))

news_lstm_test = evaluate_model(news_lstm, news_data["test_loader"], device)
news_bilstm_test = evaluate_model(news_bilstm, news_data["test_loader"], device)

news_results = pd.DataFrame([
    {"dataset": "20_newsgroups(4)", "model": "LSTM", **news_lstm_test},
    {"dataset": "20_newsgroups(4)", "model": "BiLSTM", **news_bilstm_test},
]).sort_values("f1_micro", ascending=False)

news_results

,dataset,model,f1_micro,f1_macro,f1_weighted
1,20_newsgroups(4),BiLSTM,0.343370,0.341783,0.341691
0,20_newsgroups(4),LSTM,0.252402,0.122877,0.124045


## Final comparison

In [15]:
final_comparison = pd.concat([emotion_results, news_results], ignore_index=True)
final_comparison = final_comparison.sort_values(["dataset", "f1_micro"], ascending=[True, False])
final_comparison

,dataset,model,f1_micro,f1_macro,f1_weighted
2,20_newsgroups(4),BiLSTM,0.343370,0.341783,0.341691
3,20_newsgroups(4),LSTM,0.252402,0.122877,0.124045
0,emotion,BiLSTM,0.806500,0.614334,0.774073
1,emotion,LSTM,0.391000,0.190763,0.276482
